In [1]:
import os


from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek

load_dotenv(override=True)

DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")

llm_deepseek = ChatDeepSeek(
    model='deepseek-flash',
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
)


In [ ]:
stream = input()
print(stream)

In [2]:
print("HTTP_PROXY:", os.environ.get("HTTP_PROXY"))
print("HTTPS_PROXY:", os.environ.get("HTTPS_PROXY"))
print("ALL_PROXY:", os.environ.get("ALL_PROXY"))

HTTP_PROXY: None
HTTPS_PROXY: None
ALL_PROXY: None


In [2]:
def Short_term_memory(message, max_memory = 10):
    system_message = [n for n in message if n.get("role") == "system"]
    conversation_message = [n for n in message if n.get("role") != "system"]

    recent_message = conversation_message[-(max_memory * 2):]

    return system_message + recent_message


In [3]:
long_message = [{"role":"system","content":"你是心理老师"}]
while True:
    user_input = input()
    if user_input:
        long_message.append({"role":"user","content":user_input})
    else:
        break
    long_message = Short_term_memory(long_message, max_memory = 10)
    total_chunk =''
    for chunk in llm_deepseek.stream(long_message):
        print(chunk.content,end='',flush=True)
        total_chunk = total_chunk + chunk.content
    long_message.append({"role":"assistant","content":total_chunk})


你好！很高兴见到你。作为一名心理老师，我在这里为你提供一个安全、包容的空间。无论是生活中的困惑、情绪上的波动，还是想探讨一些心理学相关的话题，都可以随时和我聊聊。你最近怎么样？有什么想分享或讨论的吗？ 🌱谢谢你愿意告诉我。学习压力大，是很真实、也很常见的感受，它不代表你不够好，也不代表你不够努力。很多时候，压力更像一个信号：它在提醒你，可能需要休息、支持，或者重新调整一下节奏。

我想先多了解你一点，这样才能更好地陪你一起看：

- 你的压力主要来自哪里？比如考试、作业、排名、父母期待，还是你对自己的要求？
- 这种压力大概持续多久了？最近有没有变得更明显？
- 它对睡眠、食欲、注意力或情绪有影响吗？
- 你现在一般会怎么应对？有没有什么事情能让你稍微喘口气？

如果你愿意，我们可以先从一个小点开始聊。也可以先试一个简单的放松：吸气4秒，停2秒，呼气6秒，做5轮。不用马上解决所有问题，我们先让你稍微松一点。

你更想先说说压力的来源，还是先找一点能立刻缓解的方法？我在这里陪你。

In [2]:
from pydantic import BaseModel, Field
from langchain_core.tools import tool
class Mytoolschema(BaseModel):
    city: str = Field(description="城市名称")


@tool(args_schema=Mytoolschema,description="查询城市气温")
def get_weather(city:str):
    result = f'{city}气温为22C'
    return result

In [3]:
deep_seek_with_tool = llm_deepseek.bind_tools([get_weather])

In [5]:
from rich import print as rprint
responses = deep_seek_with_tool.invoke("北京天气如何")

In [6]:
long_message = [{"role": "system", "content": "你是学生写的agent demo"}]

In [9]:
responses.tool_calls

[{'name': 'get_weather',
  'args': {'city': '北京'},
  'id': 'call_00_sVFqLVIOBEpYQWNJP8Ft3899',
  'type': 'tool_call'}]

In [23]:
for tool_call in responses.tool_calls:
    args = tool_call['args']
    name = tool_call['name']

In [29]:
for tool_call in responses.tool_calls:
    print(tool_call)

{'name': 'get_weather', 'args': {'city': '北京'}, 'id': 'call_00_sVFqLVIOBEpYQWNJP8Ft3899', 'type': 'tool_call'}


In [28]:
if name == 'get_weather':
    print(get_weather.invoke(args))

北京气温为22C


In [12]:
rprint(responses)

AIMessage(
    content='',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': 'The user asks about Beijing weather. I should call get_weather with city "北京".'
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 57,
            'prompt_tokens': 296,
            'total_tokens': 353,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 18,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 128},
            'prompt_cache_hit_tokens': 128,
            'prompt_cache_miss_tokens': 168
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-flash',
        'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
        'id': 'a6c01d71-85e1-446f-a4ef-c66e9fa3d9ba',
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='lc_run--01a0f23b-6dc7-7702-8546-a1ede5637c12-0',
    tool_calls=[
        {
            'name': 'get_weather',
            'args': {'city': '北京'},
            'id': 'call_00_sVFqLVIOBEpYQWNJP8Ft3899',
            'type': 'tool_call'
        }
    ],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 296,
        'output_tokens': 57,
        'total_tokens': 353,
        'input_token_details': {'cache_read': 128},
        'output_token_details': {'reasoning': 18}
    }
)

In [13]:
response2 = deep_seek_with_tool.invoke("你好啊")

In [14]:
rprint(response2)

AIMessage(
    content='你好！👋 很高兴见到你～ 
\n\n我是你的小助手，可以帮你查询各个城市的气温天气。比如你想知道北京、上海、广州今天的天气怎么样，直接告诉我就行。\
n\n有什么我能帮你的吗？😊',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': 'The user just said "你好啊" (Hello). I should respond warmly and mention what I can 
help with. I have a weather tool available. Let me respond in Chinese, friendly, and offer help.\n\nNo need to call
the tool since no city is mentioned.'
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 104,
            'prompt_tokens': 295,
            'total_tokens': 399,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 54,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 128},
            'prompt_cache_hit_tokens': 128,
            'prompt_cache_miss_tokens': 167
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-flash',
        'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
        'id': 'e30202db-6521-4555-8987-7fc3b0acf50b',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--01a0f253-7bf6-72d2-89f6-508dac0ce379-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 295,
        'output_tokens': 104,
        'total_tokens': 399,
        'input_token_details': {'cache_read': 128},
        'output_token_details': {'reasoning': 54}
    }
)

In [18]:
if not response2.tool_calls:
    print(response2.content)
else:
    print("have")

你好！👋 很高兴见到你～ 

我是你的小助手，可以帮你查询各个城市的气温天气。比如你想知道北京、上海、广州今天的天气怎么样，直接告诉我就行。

有什么我能帮你的吗？😊


In [2]:
from langchain_openai import OpenAIEmbeddings
import os
from dotenv import load_dotenv
load_dotenv("../.env", override=True)
base_url = os.getenv("PROXY_API_KEY")
api_key = os.getenv("PROXY_BASE_URL")
print(base_url)
print(api_key)


sk-ws-H.PEHEYEY.iYJB.MEQCIBe90D_Ac3uW-KgxNZxkK0aQmmw5vZLnj4Q2-XNibYMZAiAtbmt63Xlrq2ucwzO_JDay3-VBvRBP8usUMYUZk3m0kw
https://ws-1j8swy6tjowepwyg.cn-beijing.maas.aliyuncs.com/compatible-mode/v1


In [5]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    base_url=base_url,
    api_key=api_key
)

In [6]:
vector = embeddings.embed_query("测试中转配置")
print("向量维度:", len(vector))  # text-embedding-3-small 应为 1536

APIConnectionError: Connection error.

In [3]:
import os
from pathlib import Path
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

# 加载 .env 文件
load_dotenv(dotenv_path=Path(".env"))

base_url = os.getenv("PROXY_BASE_URL")
api_key = os.getenv("PROXY_API_KEY")

print("URL:", base_url)

# 关键修正：添加 check_embedding_ctx_length=False
embeddings = OpenAIEmbeddings(
    model="text-embedding-v3",
    api_key=api_key,
    base_url=base_url,
    check_embedding_ctx_length=False  # 禁止 LangChain 自行处理文本，直接发送原始字符串
)

try:
    vector = embeddings.embed_query("测试中转配置")
    print("调用成功！向量维度:", len(vector))  # text-embedding-v3 默认输出 1024 维
except Exception as e:
    print("调用失败:", e)

URL: https://ws-1j8swy6tjowepwyg.cn-beijing.maas.aliyuncs.com/compatible-mode/v1
调用成功！向量维度: 1024


In [1]:
import os

from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek

load_dotenv(override=True)

DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")

llm_deepseek = ChatDeepSeek(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    temperature=0,
    extra_body={
        "thinking": {"type": "disabled"}
    }
)

In [2]:
resp = llm_deepseek.invoke("回复一个字：好")
print(resp.response_metadata)

{'token_usage': {'completion_tokens': 1, 'prompt_tokens': 8, 'total_tokens': 9, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 8}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': 'cb0cb11b-a1cf-4e0c-97ac-dce1099b76a6', 'finish_reason': 'stop', 'logprobs': None}


In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import Literal, Optional, List
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

In [9]:
class MemoryItem(BaseModel):
    content: str = Field(description="第三人称简洁事实，如 '用户偏好本帮菜'")
    type: Literal["semantic", "procedural", "episodic"] = Field(
        description="semantic=用户事实/偏好; procedural=工具执行模式; episodic=任务经验"
    )
    durability: Literal["permanent", "long_term", "temporary", "one_off"] = Field(
        description="permanent=永久; long_term=长期可变; temporary=有期限; one_off=一次性"
    )
    key: Optional[str] = Field(
        default=None,
        description="同类事实的键，如 city、cuisine_pref；无明确归类则留空"
    )

    explicitness: Literal["explicit", "inferred"] = Field(
        description="explicit=用户明说; inferred=推断"
    )
    user_specific: bool = Field(description="是否关于该用户")
    action_relevance: Literal["high", "medium", "low"] = Field(
        description="是否影响未来行为决策"
    )

class MemoryDecision(BaseModel):
    store: bool = Field(description="是否值得长期保存")
    memories: List[MemoryItem] = Field(default_factory=list)

extractor = llm_deepseek.with_structured_output(MemoryDecision)

EXTRACT_PROMPT = """你是一个长期记忆提取器。分析用户消息，判断哪些信息值得写入长期记忆。
值得存：稳定事实、偏好、长期目标。
不要存：寒暄、一次性请求、临时状态、敏感信息。
content 用第三人称简洁表述。不要输出分数，只输出分类标签。
如果没有值得存的内容，store=false，memories=[]。
"""

In [4]:

THRESHOLD = 0.6

def compute_score(m: MemoryItem) -> float:
    SCORE_MAP = {
    "durability":       {"permanent": 1.0, "long_term": 0.8, "temporary": 0.4, "one_off": 0.0},
    "explicitness":     {"explicit": 1.0, "inferred": 0.5},
    "action_relevance": {"high": 1.0, "medium": 0.6, "low": 0.2}}
    WEIGHT = {"durability": 0.40,"explicitness": 0.25,"action_relevance": 0.20,"user_specific": 0.15}
    s = 0.0
    s += WEIGHT["durability"]       * SCORE_MAP["durability"][m.durability]
    s += WEIGHT["explicitness"]     * SCORE_MAP["explicitness"][m.explicitness]
    s += WEIGHT["action_relevance"] * SCORE_MAP["action_relevance"][m.action_relevance]
    s += WEIGHT["user_specific"]    * (1.0 if m.user_specific else 0.0)
    return round(s, 3)

In [10]:
def process(user_message: str):
    print(f"\n用户输入：{user_message}\n")

    # 调 LLM 提取
    decision = extractor.invoke([
        SystemMessage(content=EXTRACT_PROMPT),
        HumanMessage(content=f"对话内容：\n用户：{user_message}"),
    ])
    print("llm提取，并解析成功")

    if not decision.store or not decision.memories:
        print("LLM 判断：无需长期存储")
        return

    # 逐条算分 + 决定
    print("开始最终决策")
    for m in decision.memories:
        score = compute_score(m)
        passed = score >= THRESHOLD
        print(f"内容   : {m.content}")
        print(f"标签   : durability={m.durability}, explicitness={m.explicitness}, "
              f"action_relevance={m.action_relevance}, user_specific={m.user_specific}",
              f"type={m.type}, key={m.key}")
        print(f"分数   : {score}")
        print(f"决定   : {'✅ 写入' if passed else '❌ 丢弃（低于阈值）'}")
        print("-" * 50)

# ---------- 6. 测试一句 ----------
if __name__ == "__main__":
    process("我在上海")



用户输入：我在上海

llm提取，并解析成功
开始最终决策
内容   : 用户在上海
标签   : durability=long_term, explicitness=explicit, action_relevance=medium, user_specific=True type=semantic, key=city
分数   : 0.84
决定   : ✅ 写入
--------------------------------------------------
